In [ ]:
# OpenAI SDK 방식

!pip install -q openai python-dotenv
# !는 터미널 명령 실행 / openai, python-dotenv 패키지를 설치
# -q는 설치과정을 조용히(출력 최소화) 하라는 옵션
import os
# 운영체제 기능(환경변수 등)을 다루는 도구 가져오기
from dotenv import load_dotenv
# .env파일을 읽어서 환경변수로 올려주는 함수 가져오기
from openai import OpenAI
# openai 라이브러리에서 OpenAI에 접속하는 기능만 가져오기

load_dotenv()
# .env 파일에 적어둔 API키를 읽어옴
Client = OpenAI
# OpenAI에 연결하는 클라이언트 생성 / 괄호가 비어있어도 환경변수의 키를 자동 사용
Model = "gpt-4o-mini"
# 사용할 모델 이름
SYSTEM_PROMPT = "당신은 어르신입니다. 상대를 '자네'라 부르며 '~하네, ~라네, ~일세, ~하게나, ~하는가?' 같은 하게체로만 대답합니다."
# 챗봇의 성격과 역할을 정하는 지시문(system 메시지)

history = [{"role": "system", "content": SYSTEM_PROMPT}]
# 대화기록 리스트, 처음에는 system 메시지 하나만 있음
# role은 말한 사람(system/user/assistant) / content는 말한 내용

def chat(user_input, temperature=0.7, max_tokens=500):
    # chat이라는 함수 정의 / 질문(user_input)을 받아 답변을 돌려줌
    """질문을 받아 답변을 돌려주고, 대화기록에 누적한다"""
    #함수 설명서 / 실행에는 영향 없음
    history.append({"role": "user", "content": user_input})
    #사용자의 질문을 대화기록 맨 뒤에 추가
    response = client.chat.completions.create(
    # 채팅방식으로 답변을 만들어달라고 OpenAI에 요청
        model = MODEL,
        #위에서 정한 모델
        messages = history,
        #지금까지의 대화 전체를 보내 이전 내용을 기억하는 것처럼 동작
        temperature = temperature,
        #temperature 값 전달
        max_tokens = max_tokens
        # 최대 토큰 수 전달
    )
    answer = response.choices[0].message.content
    # 응답에서 답변 텍스트만 꺼냄
    # choices: 답변 후보 목록 / [0] : 첫번째 후보 / message.content: 본문
    history.append({"role": "assistant", "content": answer})
    # 챗봇의 답변도 기록에 추가, 다음 질문에서 자기 답변을 기억함

def reset():
    # 대화 기록을 처음상태로 되돌리는 함수
    """대화 기록 초기화"""
    history.clear()
    # 리스트 안의 내용을 전부 비움
    history.append({"role": "system", "content": SYSTEM_PROMPT})
    # system 메시지만 다시 넣어 처음 상태로 복구

print("챗봇 답변 준비 완료")

챗봇 답변 준비 완료



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
question = ""
#질문을 변수에 담음. 문장만 바꿔서 실행하면 됨
print("질문:", question)
# 어떤 질문을 했는지 표시
print("답변:", chat(question))
#chat 함수에 질문을 넣고, 돌려받은 답변 출력

In [10]:
# LangChain 방식

!pip install -q langchain langchain-openai langchain-core python-dotenv
# 필요한 도구를 설치하는 명령(한 번 설치했으면 다음부터는 지워도 됨)
# langchain : 챗봇 부품을 조립하게 해주는 도구 모음
# langchain-openai : LangChain에서 OpenAI를 쓰기위한 연결부품

from dotenv import load_dotenv
# .env 파일에 적어둔 API 키를 읽어오는 기능

from langchain_openai import ChatOpenAI
# 부품 1. OpenAI 모델을 LangChain에서 쓰게 해주는 클래스

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 부품 2. 프롬프트 템플릿(질문 틀)과 그 틀안에 대화기록 자리를 만들어주는 도구

from langchain_core.messages import HumanMessage, AIMessage
# 대화 기록에 저장할 '내 말' / 'AI의 말' 형태의 메시지 객체

load_dotenv()
# .env 파일의 OPENAI_API_KEY를 읽어서 컴퓨터에 올려둠

llm = ChatOpenAI(model= "gpt-4o-mini", temperature = 0.7, max_tokens= 500)
# 부품 1: 모델 / 사용할 AI를 준비

PERSONA =  "당신은 어르신입니다. 상대를 '자네'라 부르며 '~하네, ~라네, ~일세, ~하게나, ~하는가?' 같은 하게체로만 대답합니다."
# AI의 역할을 정하는 부분

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {persona}입니다."),
    # AI에게 주는 역할 설정
    
    MessagesPlaceholder("history"),
    # 이전 대화가 들어갈 빈자리 / 지금까지의 대화 기록이 들어옴

    ("human", "{question}"),
    # 이번에 새로하는 질문 / {question} 자리에 나중에 질문이 들어감
])
# 부품 2: 프롬프트 / 질문 틀(템플릿)을 만듬 / 중괄호{}는 나중에 채워질 빈칸

chain = prompt | llm
# 부품 3: 틀| 모델 = 틀에 값을 채우고 그 결과를 모델에게 넘김
# |는 왼쪽 결과를 오른쪽으로 전달하는 파이프

history = []
# 대화 메모장, 처음에는 비어있음

def chat(question):
    # 질문을 받아 답변을 돌려주는 함수
    result = chain.invoke({
        # invoke: 체인을 실행 / 딕셔너리의 키 이름이 템플릿의 빈칸, 이름과 정확히 같아야함
        "persona": PERSONA,      # 템플릿의 {persona} 빈칸에 들어갈 값
        "history": history,     # 이전 대화기록 / (MessagesPlaceholder 자리에 들어감)
        "question": question,    # 템플릿의 {question} 빈칸에 들어갈 값
})

    answer = result.content
    # 응답에서 답변 글자만 꺼냄(LangChain은 .content 한단계로 가능)

    history.append(HumanMessage(question))
    # 메모장에 '내 질문'을 기록

    history.append(AIMessage(answer))
    #메모장에 'AI의 답변'도 기록 --> 다음 질문 때 이전 대화를 기억할 수 있음

    return answer
    # 답변을 함수 밖으로 돌려줌

def reset():
    # 대화 기록을 지우고 새로 시작하는 함수
    history.clear()
    # 메모장을 비움

print("LangChain 챗봇 답변 준비 완료")
# 셀이 끝까지 잘 실행됐는지 확인

LangChain 챗봇 답변 준비 완료



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
question = ""
# 질문 입력 / 문장만 바꿔서 다시 실행하면 됨

print("질문:", question)
# 어떤 질문을 했는지

print("답변:", chat(question))
# 위의 chat 함수에 질문을 넘기고, 돌아온 답변 출력

In [ ]:
# 글자가 실시간으로 나오게 하기

for chunk in chain.stream({"persona":PERSONA, "history": history, "question": question}):
    print(chunk.content, end="", flush = True)
# chain.stream = 답변을 조각단위로 보내달라는 요청
#{"persona"}... = 템플릿의 빈칸 3개를 채울 값. 키 이름이 템플릿 변수와 같아야함
# chunk = 도착한 조각 하나
# for..in... = 조각이 올때마다 아래줄 반복 실행, 더 올 조각이 없으면 자동 종료
# chunk.content = 조각안에 있는 글자만 꺼냄
# end="" 줄바꿈 대신 빈 글자를 붙이라는 뜻 / 없으면 답이 세로로 길게 나옴
# flush = True --> 출력할 글자를 모아두지 말고 바로 화면으로 내보내라는 뜻 

# invoke() = 답변이 한번에 나오게
# stream() = ai가 글자를 만들어내는대로 조금씩 받아서 화면에 찍음

# 이 코드는 답변을 화면에만 보여주고 history에 기록하지 않음
# 기억하게 하려면 조각을 모아두었다가 끝에 기록해야 함